# BERT by hand: the encoder

This is the homework for Part 1 of **BERT by Hand**. You'll push one tiny masked sentence pair through a BERT encoder layer and **write down every matrix yourself**.

**How it works**

1. Run the first code cell. It sets up the pair, a tiny vocabulary and all the weights.
2. Each step shows a code cell full of `nan`. Replace every `nan` with your number, then run the cell. It checks your answer straight away.
3. If something is off, the checker says which entries and gives a hint for the first one. Still stuck? Call `reveal('name')` to see the answer and carry on.

Work on paper or use Python as a calculator for single numbers (`np.exp(0.5)`, `np.tanh(1)`, `0.3*2 + 1.1*0.5`). Try not to use `@` on whole matrices: doing the dot products yourself is the point. Two decimals is enough. Each check uses *your* previous checked matrices, so small rounding differences won't snowball.

**The example.** Sentence A is "dog", sentence B is "barks". Packed, that's `[CLS] dog [SEP] barks [SEP]`, and "dog" was picked for masking, so the model reads `[CLS] [MASK] [SEP] barks [SEP]`. B really follows A, so the NSP label is IsNext.

**The sizes we use**

| setting | here | BERT-base |
|---|---|---|
| tokens n | 5 | up to 512 |
| hidden size H | 4 | 768 |
| heads A | 2 | 12 |
| d_k = H / A | 2 | 64 |
| d_ff | 8 | 3072 |
| layers L | 1 | 12 |
| vocabulary V | 8 | 30,000 |

Dropout is skipped. A few details (the LayerNorm on the embeddings, the GELU formula) come from Google's released code rather than the paper; the site marks them with a *code* tag.

In [ ]:
# Run this cell first. It sets up the example, the weights and the checker.
# You don't need to read it, and please don't peek at REF: that's the answer key.
import numpy as np
np.set_printoptions(precision=4, suppress=True)
nan = np.nan

# ---- settings for this run (BERT-base values in comments) ----
sentence_A = "dog"      # the word "dog" is hidden behind [MASK]
sentence_B = "barks"
tokens   = ["[CLS]", "[MASK]", "[SEP]", "barks", "[SEP]"]
mask_pos = 1            # position of the hidden word
target   = "dog"        # what the MLM head must predict there
label    = 0            # 0 = IsNext (B really follows A), 1 = NotNext
n    = 5    # tokens              (BERT: up to 512)
H    = 4    # hidden size         (BERT-base: 768)
A    = 2    # attention heads     (BERT-base: 12)
d_k  = H // A   # = 2             (BERT-base: 64)
d_ff = 8    # FFN inner width     (BERT-base: 3072 = 4H)
V    = 8    # vocabulary size     (BERT: 30,000 WordPiece pieces)

# ---- a tiny WordPiece vocabulary and its (made-up) learned token table, V x H ----
vocab = {"[PAD]": 0, "[CLS]": 1, "[SEP]": 2, "[MASK]": 3, "dog": 4, "cat": 5, "barks": 6, "sleeps": 7}
W_tok = np.array([
    [ 0.0,  0.0,  0.0,  0.0],   # 0 [PAD]
    [ 0.2,  0.2,  0.5,  0.4],   # 1 [CLS]
    [ 0.4, -0.3,  0.3, -0.2],   # 2 [SEP]
    [-0.4,  0.5, -0.2, -0.5],   # 3 [MASK]
    [-0.2,  0.5,  0.4, -0.5],   # 4 dog
    [ 0.0,  0.1,  0.0, -0.4],   # 5 cat
    [ 0.5, -0.5,  0.3, -0.2],   # 6 barks
    [ 0.5, -0.5, -0.3,  0.2],   # 7 sleeps
])
# ---- segment table (row 0: sentence A, row 1: sentence B) and LEARNED position table, n x H ----
W_seg = np.array([[-0.3, -0.4,  0.3,  0.5],
                  [ 0.2,  0.2,  0.3, -0.4]])
W_pos = np.array([
    [ 0.2, -0.1,  0.5,  0.4],   # position 0
    [ 0.0, -0.4, -0.5,  0.5],   # position 1
    [-0.5, -0.2, -0.5, -0.1],   # position 2
    [-0.5,  0.4,  0.3,  0.1],   # position 3
    [ 0.3,  0.2, -0.4,  0.2],   # position 4
])
eps = 1e-12   # LayerNorm's tiny safety constant in the released code

# ---- attention weights (made up; a trained model learns these), each H x d_k ----
W_Q1 = np.array([[ 0.0,  0.5],
                 [ 1.0,  0.0],
                 [ 0.0, -1.0],
                 [ 1.0, -1.0]])
W_K1 = np.array([[ 1.0,  0.5],
                 [-0.5,  0.0],
                 [ 0.5,  0.5],
                 [ 0.0,  0.5]])
W_V1 = np.array([[-1.0,  0.5],
                 [ 0.5,  0.0],
                 [ 0.0,  1.0],
                 [ 0.0,  0.5]])
W_Q2 = np.array([[ 1.0, -0.5],
                 [ 1.0,  0.5],
                 [ 0.5,  0.5],
                 [ 0.5,  0.5]])
W_K2 = np.array([[ 0.5,  0.0],
                 [-1.0, -1.0],
                 [ 1.0,  1.0],
                 [-1.0,  0.0]])
W_V2 = np.array([[ 0.0,  1.0],
                 [-0.5, -0.5],
                 [-1.0,  0.0],
                 [-1.0,  0.5]])
W_O = np.array([[-1.0, -1.0,  1.0,  0.5],
                [ 0.0,  0.0, -0.5, -0.5],
                [ 1.0,  0.5,  0.5, -1.0],
                [ 0.5,  0.0,  0.0,  0.5]])

# ---- feed-forward weights: W1 is H x d_ff, W2 is d_ff x H ----
W1 = np.array([[ 0.5,  1.0, -0.5,  1.0,  0.5, -0.5,  1.0,  0.5],
               [-0.5,  0.5,  1.0,  1.0,  1.0, -1.0,  0.0,  0.5],
               [ 0.5, -0.5, -1.0, -1.0, -1.0,  0.0, -0.5,  0.0],
               [ 0.5,  0.0,  1.0,  0.0,  0.5, -0.5, -1.0,  0.0]])
b1 = np.array([ 0.0,  0.1, -0.1, -0.1,  0.1,  0.0,  0.0,  0.0])
W2 = np.array([[-0.5, -1.0,  0.0,  0.0],
               [ 0.0,  0.0,  0.0,  1.0],
               [-0.5, -1.0, -1.0,  0.0],
               [-0.5, -0.5, -1.0,  1.0],
               [-1.0,  0.5,  1.0,  0.0],
               [-1.0, -1.0, -1.0,  0.0],
               [-0.5,  0.0,  1.0,  0.5],
               [ 1.0,  0.0,  0.0,  0.0]])
b2 = np.array([ 0.0, -0.1,  0.0,  0.1])

# =============== the checker (no need to read below) ===============
def _sm(S):
    e = np.exp(S - S.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)
def _ln(R):
    mu = R.mean(axis=1); sd = np.sqrt(R.var(axis=1) + eps)
    return mu, sd, (R - mu[:, None]) / sd[:, None]
def _gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))

_ok = {}
def got(k): return _ok[k] if k in _ok else REF[k]

_RULES = {   # how each matrix is computed from the matrices before it
  "ids":  lambda: np.array([vocab[t] for t in tokens], dtype=float),
  "seg":  lambda: np.array([0, 0, 0, 1, 1], dtype=float),
  "E_tok": lambda: W_tok[got("ids").astype(int)],
  "E_seg": lambda: W_seg[got("seg").astype(int)],
  "E_pos": lambda: W_pos[:n],
  "X_sum": lambda: got("E_tok") + got("E_seg") + got("E_pos"),
  "mu0":  lambda: _ln(got("X_sum"))[0], "sd0": lambda: _ln(got("X_sum"))[1],
  "X0":   lambda: (got("X_sum") - got("mu0")[:, None]) / got("sd0")[:, None],
  "Q1":   lambda: got("X0") @ W_Q1, "K1": lambda: got("X0") @ W_K1, "V1": lambda: got("X0") @ W_V1,
  "Q2":   lambda: got("X0") @ W_Q2, "K2": lambda: got("X0") @ W_K2, "V2": lambda: got("X0") @ W_V2,
  "S1":   lambda: got("Q1") @ got("K1").T, "S2": lambda: got("Q2") @ got("K2").T,
  "S1s":  lambda: got("S1") / np.sqrt(d_k), "S2s": lambda: got("S2") / np.sqrt(d_k),
  "A1":   lambda: _sm(got("S1s")), "A2": lambda: _sm(got("S2s")),
  "H1":   lambda: got("A1") @ got("V1"), "H2": lambda: got("A2") @ got("V2"),
  "C":    lambda: np.concatenate([got("H1"), got("H2")], axis=1),
  "MHA":  lambda: got("C") @ W_O,
  "R1":   lambda: got("X0") + got("MHA"),
  "mu1":  lambda: _ln(got("R1"))[0], "sd1": lambda: _ln(got("R1"))[1],
  "X1":   lambda: (got("R1") - got("mu1")[:, None]) / got("sd1")[:, None],
  "P":    lambda: got("X1") @ W1 + b1,
  "I":    lambda: _gelu(got("P")),
  "F":    lambda: got("I") @ W2 + b2,
  "R2":   lambda: got("X1") + got("F"),
  "mu2":  lambda: _ln(got("R2"))[0], "sd2": lambda: _ln(got("R2"))[1],
  "T":    lambda: (got("R2") - got("mu2")[:, None]) / got("sd2")[:, None],
}
REF = {}
for _k, _f in _RULES.items():   # the full answer key, computed in order
    REF[_k] = _f()

_HINTS = {
  "ids": "Look each token up in the vocab dictionary. Position 1 is [MASK], not dog.",
  "seg": "0 from [CLS] up to and including the first [SEP], then 1.",
  "E_tok": "Row r is the row of W_tok whose number is ids[r]. Just copy the row.",
  "E_seg": "Row r is W_seg[seg[r]]: row 0 for sentence A, row 1 for sentence B.",
  "E_pos": "Row r is W_pos[r]. BERT learns these rows; no sines or cosines.",
  "X_sum": "Add the three tables entry by entry. No scaling by sqrt(H) in BERT.",
  "mu0": "The mean of each row of X_sum (sum the 4 numbers, divide by 4).",
  "sd0": "For each row: average the squared distances from the row's mean, then take the square root.",
  "X0": "(X_sum[r, c] - mu0[r]) / sd0[r]. This LayerNorm comes from the released code.",
  "S1s": "Divide every score by sqrt(d_k) = sqrt(2) ≈ 1.4142.",
  "S2s": "Divide every score by sqrt(d_k) = sqrt(2) ≈ 1.4142.",
  "A1": "For each row: take e**x of every entry, then divide each by the row's total. Each row must sum to 1.",
  "A2": "For each row: take e**x of every entry, then divide each by the row's total. Each row must sum to 1.",
  "C": "Put H1 and H2 side by side: the first d_k columns are H1, the next d_k are H2.",
  "R1": "Residual: add the layer's input X0 to MHA, entry by entry.",
  "mu1": "The mean of each row of R1.",
  "sd1": "The standard deviation of each row of R1 (divide by 4, not 3).",
  "X1": "(R1[r, c] - mu1[r]) / sd1[r].",
  "P": "X1 @ W1, then add b1[c] to every entry in column c.",
  "I": "GELU(x) = 0.5*x*(1 + tanh(0.7979*(x + 0.044715*x**3))). Use np.tanh for single numbers.",
  "F": "I @ W2, then add b2[c] to every entry in column c.",
  "R2": "Residual again: X1 + F, entry by entry.",
  "mu2": "The mean of each row of R2.",
  "sd2": "The standard deviation of each row of R2.",
  "T": "(R2[r, c] - mu2[r]) / sd2[r]. Row 0 is C, the [CLS] vector.",
}
_MATMUL = {"Q1": ("X0", W_Q1, "W_Q1"), "K1": ("X0", W_K1, "W_K1"), "V1": ("X0", W_V1, "W_V1"),
           "Q2": ("X0", W_Q2, "W_Q2"), "K2": ("X0", W_K2, "W_K2"), "V2": ("X0", W_V2, "W_V2"),
           "S1": ("Q1", None, "K1ᵀ"), "S2": ("Q2", None, "K2ᵀ"),
           "H1": ("A1", None, "V1"), "H2": ("A2", None, "V2"), "MHA": ("C", W_O, "W_O")}

def _hint(name, r, c):
    if name in _MATMUL:
        a, B, bname = _MATMUL[name]
        A_ = got(a)
        if B is None:
            B = {"S1": lambda: got("K1").T, "S2": lambda: got("K2").T,
                 "H1": lambda: got("V1"), "H2": lambda: got("V2")}[name]()
        terms = " + ".join(f"({A_[r, j]:.2f})×({B[j, c]:.2f})" for j in range(A_.shape[1]))
        return f"{name}[{r},{c}] = row {r} of {a} · column {c} of {bname} = {terms}"
    return _HINTS.get(name, "")

def check(name, yours):
    """Grade your matrix. Wrong or blank cells are listed with a hint for the first one."""
    exp = np.atleast_1d(np.asarray(_RULES[name](), dtype=float))
    y = np.atleast_1d(np.asarray(yours, dtype=float))
    if y.shape != exp.shape:
        print(f"✗ {name}: shape {y.shape}, but it should be {exp.shape}."); return
    if np.isnan(y).all():
        print(f"… {name} is still blank. Replace every nan with your number."); return
    tol = 0.02 + 0.01 * np.abs(exp)
    bad = np.argwhere(np.isnan(y) | (np.abs(y - exp) > tol))
    if len(bad) == 0:
        _ok[name] = y if np.ndim(yours) else float(y[0])
        print(f"✓ {name}{' ' + str(exp.shape) if np.ndim(yours) else ''} is correct. Nice work!"); return
    print(f"✗ {name}: {len(bad)} of {exp.size} entries need another look:")
    for idx in bad[:6]:
        where = f"[{', '.join(map(str, idx))}]"
        print(f"   {name}{where}: you have {y[tuple(idx)]:.4g}")
    idx = bad[0]
    r, c = (int(idx[0]), int(idx[1])) if y.ndim == 2 else (int(idx[0]), 0)
    print("   Hint:", _hint(name, r, c))
    print(f"   Stuck? reveal('{name}') shows the answer and lets you carry on.")

def reveal(name):
    """Show the answer for one matrix and use it from here on."""
    exp = _RULES[name](); _ok[name] = exp
    print(f"{name} {np.shape(exp)} =\n{np.round(exp, 4)}")

def check_shapes(answers):
    """Grade a dictionary of shapes for BERT-base and a 128-token input."""
    PAPER = {"E_tok": (128, 768), "W_Q1": (768, 64), "Q1": (128, 64), "S1": (128, 128), "A1": (128, 128),
             "H1": (128, 64), "C": (128, 768), "W_O": (768, 768), "P": (128, 3072), "T": (128, 768)}
    right = 0
    for k, v in PAPER.items():
        mine = tuple(answers.get(k, ()))
        ok = mine == v; right += ok
        print(f"{'✓' if ok else '✗'} {k:6s} you said {mine or '(blank)'}" + ("" if ok else f", BERT-base has {v}"))
    print(f"\n{right} of {len(PAPER)} right.")

print("Ready. Tokens:", tokens, "| n =", n, "| H =", H, "| A =", A, "| d_k =", d_k, "| d_ff =", d_ff, "| V =", V)

## Step 1: Input IDs and segment IDs

Look each of the 5 tokens up in `vocab`. Then write the segment ID of each position: 0 for `[CLS]`, sentence A and the first `[SEP]`; 1 for sentence B and the last `[SEP]`.

**Shape:** (n,) = (5,) each

In [ ]:
# one ID per token, [CLS] [MASK] [SEP] barks [SEP]
ids = np.array([nan, nan, nan, nan, nan])
check('ids', ids)

In [ ]:
# 0 or 1 per token
seg = np.array([nan, nan, nan, nan, nan])
check('seg', seg)

## Step 2: Three lookups

BERT's input is built from three tables, each with H = 4 columns:

- `E_tok`: row `ids[r]` of the token table `W_tok` (8 × 4 here, 30,000 × 768 in BERT-base)
- `E_seg`: row `seg[r]` of the segment table `W_seg` (2 × 4)
- `E_pos`: row r of the **learned** position table `W_pos` (512 × 768 in BERT-base, no sines)

**Shape:** (n × H) = (5 × 4) each

In [ ]:
print('W_tok =\n', W_tok, '\nW_seg =\n', W_seg, '\nW_pos =\n', W_pos)

In [ ]:
# row r = W_tok[ids[r]]; row 1 is the [MASK] row
E_tok = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('E_tok', E_tok)

In [ ]:
# row r = W_seg[seg[r]]
E_seg = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('E_seg', E_seg)

In [ ]:
# row r = W_pos[r]
E_pos = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('E_pos', E_pos)

## Step 3: Add them, then LayerNorm

`X_sum = E_tok + E_seg + E_pos`. Unlike the Transformer, there's no × √H scaling. The released code then normalises each row before the first layer: subtract the row's mean, divide by its standard deviation (γ = 1, β = 0 here).

In [ ]:
# entry by entry
X_sum = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('X_sum', X_sum)

In [ ]:
# mean of each row of X_sum
mu0 = np.array([nan, nan, nan, nan, nan])
check('mu0', mu0)

In [ ]:
# standard deviation of each row (divide by 4)
sd0 = np.array([nan, nan, nan, nan, nan])
check('sd0', sd0)

In [ ]:
# (X_sum[r, c] - mu0[r]) / sd0[r]
X0 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('X0', X0)

## Step 4: Queries, keys and values for each head

From here on the layer is the Transformer encoder you already know. Each head has its own W_Q, W_K and W_V, each (H × d_k) = (4 × 2):

(5 × 4) · (4 × 2) = (5 × 2)

### Head 1

In [ ]:
print('W_Q1 =\n', W_Q1, '\nW_K1 =\n', W_K1, '\nW_V1 =\n', W_V1)

In [ ]:
# X0 · W_Q1
Q1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('Q1', Q1)

In [ ]:
# X0 · W_K1
K1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('K1', K1)

In [ ]:
# X0 · W_V1
V1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('V1', V1)

### Head 2

In [ ]:
print('W_Q2 =\n', W_Q2, '\nW_K2 =\n', W_K2, '\nW_V2 =\n', W_V2)

In [ ]:
# X0 · W_Q2
Q2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('Q2', Q2)

In [ ]:
# X0 · W_K2
K2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('K2', K2)

In [ ]:
# X0 · W_V2
V2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('V2', V2)

## Step 5: Scores, scale, softmax

`S = Q · Kᵀ` is (5 × 2) · (2 × 5) = (5 × 5). There is **no mask**: `[MASK]` at position 1 can look at "barks" to its right as well as `[CLS]` to its left. That's the "bidirectional" in BERT's name. Then divide by √d_k = √2 and softmax each row.

In [ ]:
# head 1: S1[r, c] = Q1[r] · K1[c]
S1 = np.array([
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan]
])
check('S1', S1)

In [ ]:
# head 2
S2 = np.array([
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan]
])
check('S2', S2)

In [ ]:
# S1 / sqrt(2)
S1s = np.array([
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan]
])
check('S1s', S1s)

In [ ]:
# S2 / sqrt(2)
S2s = np.array([
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan]
])
check('S2s', S2s)

In [ ]:
# head 1 attention weights, rows sum to 1
A1 = np.array([
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan]
])
check('A1', A1)

In [ ]:
# head 2 attention weights
A2 = np.array([
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan]
])
check('A2', A2)

## Step 6: Blend the values, join the heads

`H = A · V` is (5 × 5) · (5 × 2) = (5 × 2). Then put the heads side by side, (5 × 4), and multiply by W_O (4 × 4).

In [ ]:
# A1 · V1
H1 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('H1', H1)

In [ ]:
# A2 · V2
H2 = np.array([
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan],
    [nan, nan]
])
check('H2', H2)

In [ ]:
# [H1 | H2]
C = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('C', C)

In [ ]:
print(W_O)

In [ ]:
# C · W_O
MHA = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('MHA', MHA)

## Step 7: Add & Norm (1)

In [ ]:
# X0 + MHA
R1 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('R1', R1)

In [ ]:
# mean of each row of R1
mu1 = np.array([nan, nan, nan, nan, nan])
check('mu1', mu1)

In [ ]:
# standard deviation of each row of R1
sd1 = np.array([nan, nan, nan, nan, nan])
check('sd1', sd1)

In [ ]:
# (R1[r, c] - mu1[r]) / sd1[r]
X1 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('X1', X1)

## Step 8: Feed-forward with GELU

The one change inside the layer: BERT uses **GELU** instead of ReLU (paper appendix A.2). The released code uses this formula:

`GELU(x) = 0.5 · x · (1 + tanh(√(2/π) · (x + 0.044715 · x³)))`, with √(2/π) ≈ 0.7979.

Big positive numbers pass almost unchanged, big negative ones become almost 0, and small negatives stay slightly negative.

(5 × 4) · (4 × 8) = (5 × 8), then (5 × 8) · (8 × 4) = (5 × 4)

In [ ]:
print('W1 =\n', W1, '\nb1 =', b1)

In [ ]:
# X1 · W1 + b1
P = np.array([
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan]
])
check('P', P)

In [ ]:
# GELU(P), entry by entry
I = np.array([
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan]
])
check('I', I)

In [ ]:
print('W2 =\n', W2, '\nb2 =', b2)

In [ ]:
# I · W2 + b2
F = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('F', F)

## Step 9: Add & Norm (2), the encoder output T

In [ ]:
# X1 + F
R2 = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('R2', R2)

In [ ]:
# mean of each row of R2
mu2 = np.array([nan, nan, nan, nan, nan])
check('mu2', mu2)

In [ ]:
# standard deviation of each row of R2
sd2 = np.array([nan, nan, nan, nan, nan])
check('sd2', sd2)

In [ ]:
# (R2[r, c] - mu2[r]) / sd2[r]
T = np.array([
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan],
    [nan, nan, nan, nan]
])
check('T', T)

## You did it

T is the output of one BERT encoder layer: one 4-number vector per token, each mixed with context from both sentences and from both directions. BERT-base repeats this 12 times, each layer with its own weights.

Run the cell below to see your whole pass, step by step.

In [ ]:
for k in ["ids", "E_tok", "X_sum", "X0", "Q1", "S1", "A1", "H1", "C", "MHA", "X1", "P", "I", "F", "T"]:
    print(f"{k:9s} {str(np.shape(got(k))):8s} {'✓ yours' if k in _ok else '· answer key'}")
print(f"\n{len(_ok)} of {len(_RULES)} matrices done by you.")

## Bonus: BERT-base sizes, by hand

No numbers this time, just shapes. Imagine a 128-token input and BERT-base (H 768, A 12, d_k 64, d_ff 3072). Fill in each shape as a tuple, like `(128, 768)`.

In [ ]:
paper_shapes = {
    "E_tok": (),   # token embeddings after the lookup
    "W_Q1":  (),   # one head's query weights
    "Q1":    (),   # one head's queries
    "S1":    (),   # one head's scores
    "A1":    (),   # one head's attention weights
    "H1":    (),   # one head's output
    "C":     (),   # all heads concatenated
    "W_O":   (),   # output projection
    "P":     (),   # FFN hidden layer
    "T":     (),   # encoder output
}
check_shapes(paper_shapes)

## Want more?

- **Mask a different word.** In the first cell, set `tokens` to `["[CLS]", "dog", "[SEP]", "[MASK]", "[SEP]"]`, `mask_pos = 3` and `target = "barks"`, then redo the pass. The checker recomputes the answer key.
- **Make the weights your own.** Edit any weight matrix and rerun.
- **Compare with the site.** Every step here matches a step in BERT by Hand, which also animates each multiplication.